# 07 — Analyze Card Evaluation

Replicates the **Analyze Card** Chrome extension pipeline locally so you can test on cards with known PSA grades.

**Pipeline:**
1. Load local card images (front + back)
2. Send to Claude Vision with the same prompt used in production
3. Display grade estimate, probability bars, and issues
4. Compare against your known ground-truth grade

In [1]:
# ── Setup ────────────────────────────────────────────────────────
import os, base64, json, textwrap
from pathlib import Path
from dotenv import load_dotenv
import anthropic
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np

# Load API key from .env.local (fill in ANTHROPIC_API_KEY there, or set below)
load_dotenv('../.env.local')
API_KEY = os.getenv('ANTHROPIC_API_KEY') or ''

if not API_KEY:
    raise ValueError('Set ANTHROPIC_API_KEY in .env.local or directly above')

client = anthropic.Anthropic(api_key=API_KEY)
MODEL  = 'claude-haiku-4-5'   # same as production; change to sonnet for higher accuracy

ROOT = Path('.').resolve()   # notebooks/
print(f'Working dir : {ROOT}')
print(f'Model       : {MODEL}')
print('API key     : ✓ loaded')

Working dir : /Users/srinivasdoddi/srini/agentic-card-seller-os/notebooks
Model       : claude-haiku-4-5
API key     : ✓ loaded


In [2]:
# ── Prompts (identical to production claudeVision.ts) ────────────

SYSTEM_PROMPT = """You are an expert PSA card grader specializing in Pokémon trading cards.
Your job is to examine card images and estimate what PSA grade the card would receive.

PSA Grade Reference:
- PSA 10 Gem Mint:   Perfect centering (55/45 or better), four razor-sharp corners, no print defects, full original gloss
- PSA 9  Mint:       Near-perfect, slight wear allowed on 1-2 corners, well-centred, minor print spots
- PSA 8  NM-MT:      Light wear on 3+ corners or edges, may have very light scratches
- PSA 7  NM:         Slight corner/edge wear visible, possible light scratches, minor print defects
- PSA 6  EX-MT:      Moderate corner wear, light scratches, slight edge fraying
- PSA 5  EX:         Heavy corner wear, noticeable scratches, obvious edge nicks
- PSA 4  VG-EX:      Obvious wear throughout, possible light crease
- PSA 3  VG:         Heavy wear, possible creases, staining
- PSA 2  Good:       Creases, heavy staining, major defects
- PSA 1  Poor:       Severe damage, missing pieces, heavy creases

Examine these areas in order:
1. CORNERS — check each corner (TL, TR, BL, BR) for whitening, fraying, rounding
2. EDGES   — check all four edges for nicks, chips, whitening
3. CENTERING — estimate left/right and top/bottom split (e.g. "60/40 L/R")
4. SURFACE — scratches, print lines, holo damage, staining, indentations
5. CARD IDENTITY — read the card name, set, year, and collector number from the image

Important notes:
- If only one image is provided assume it is the front; note missing back reduces confidence
- eBay listing photos are often taken at an angle or with slight glare; account for this
- Pokémon cards from 1999-2003 (Base Set through Skyridge) are printed differently from modern cards; adjust expectations accordingly
- Holo cards must have intact holo pattern with no scratches to achieve PSA 9+

Respond ONLY with a single valid JSON object — no markdown, no explanation, no extra text."""

USER_PROMPT_TEMPLATE = """Analyze this Pokémon card and return your assessment as JSON with exactly this structure:

{{
  "card_identity": {{"name": "Charizard", "set": "Base Set", "year": "1999", "number": "4"}},
  "grade_estimate": {{
    "grade_range":  "PSA 7-8",
    "confidence":   "high",
    "distribution": {{"1":0.00,"2":0.00,"3":0.01,"4":0.02,"5":0.04,"6":0.08,"7":0.30,"8":0.40,"9":0.12,"10":0.03}}
  }},
  "issues": ["Light corner whitening on top-left", "Minor edge nick on right edge"],
  "image_quality": {{"status": "good", "warnings": []}}
}}

Listing title (use as identity hint, trust image over title): "{title}"

Rules:
- distribution values must sum to 1.0
- confidence is high when top-2 PSA grades account for >60% probability
- confidence is medium when top-2 grades account for 40-60%
- confidence is low when image is blurry, small, or too angled to assess
- issues should be specific and actionable (mention which corner/edge)
- if the back image is missing add \"back image missing — confidence reduced\" to warnings"""

In [3]:
# ── Helper functions ─────────────────────────────────────────────

def image_to_base64(path: str | Path) -> str:
    """Read a local image file and return base64-encoded string."""
    with open(path, 'rb') as f:
        return base64.standard_b64encode(f.read()).decode('utf-8')

def analyze_card(image_paths: list[str | Path], title: str = '') -> dict:
    """
    Run the same Claude Vision grading pipeline as the Chrome extension.
    Accepts local file paths (converted to base64) or https:// URLs.
    """
    content = []

    for p in image_paths[:6]:
        p = str(p)
        if p.startswith('http'):
            content.append({'type': 'image', 'source': {'type': 'url', 'url': p}})
        else:
            b64 = image_to_base64(p)
            content.append({
                'type': 'image',
                'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b64},
            })

    content.append({
        'type': 'text',
        'text': USER_PROMPT_TEMPLATE.format(title=title or 'Unknown card'),
    })

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': content}],
    )

    raw = response.content[0].text.strip()
    raw = raw.removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    result = json.loads(raw)

    # Normalise distribution to sum=1
    dist  = result['grade_estimate']['distribution']
    total = sum(dist.values())
    if total > 0 and abs(total - 1) > 0.02:
        for k in dist:
            dist[k] = round(dist[k] / total, 4)

    # Attach token usage
    result['_usage'] = {
        'input_tokens':  response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
    }
    return result


def estimate_cost(usage: dict, model: str = MODEL) -> float:
    rates = {
        'claude-haiku-4-5':  (0.80, 4.00),
        'claude-sonnet-4-5': (3.00, 15.00),
        'claude-opus-4-5':   (15.00, 75.00),
    }
    inp, out = rates.get(model, (3.00, 15.00))
    return (usage['input_tokens'] / 1e6 * inp) + (usage['output_tokens'] / 1e6 * out)


print('Helper functions defined ✓')

Helper functions defined ✓


In [4]:
# ── Visualisation ────────────────────────────────────────────────

GRADE_COLORS = {
    10: '#059669', 9: '#10b981', 8: '#34d399',
    7:  '#fbbf24', 6: '#f59e0b',
    5:  '#f97316', 4: '#ef4444', 3: '#dc2626', 2: '#b91c1c', 1: '#7f1d1d',
}
DECISION_COLORS = {'buy': '#059669', 'maybe': '#d97706', 'skip': '#dc2626'}

def display_result(result: dict, image_paths: list, known_grade: int | None = None):
    """Render a card analysis result with images and grade bars."""
    identity = result.get('card_identity', {})
    est      = result.get('grade_estimate', {})
    dist     = est.get('distribution', {})
    issues   = result.get('issues', [])
    quality  = result.get('image_quality', {})
    usage    = result.get('_usage', {})
    cost     = estimate_cost(usage)

    n_imgs = min(len(image_paths), 2)
    fig = plt.figure(figsize=(16, 8))
    gs  = gridspec.GridSpec(1, 3, width_ratios=[1]*n_imgs + [2.5], wspace=0.3)

    # ── Card images ──
    for i, p in enumerate(image_paths[:2]):
        ax = fig.add_subplot(gs[i])
        try:
            img = Image.open(p)
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Image\nnot found', ha='center', va='center',
                    transform=ax.transAxes, color='gray')
        ax.axis('off')
        ax.set_title(['Front', 'Back'][i], fontsize=11, color='#6b7280')

    # ── Results panel ──
    ax_r = fig.add_subplot(gs[n_imgs])
    ax_r.axis('off')

    # Title
    name = identity.get('name', '?')
    cset = identity.get('set',  '')
    year = identity.get('year', '')
    num  = identity.get('number', '')
    card_label = f"{name}" + (f" — {cset}" if cset else '') + (f" ({year})" if year else '') + (f" #{num}" if num else '')
    ax_r.text(0.0, 1.02, card_label, transform=ax_r.transAxes,
              fontsize=13, fontweight='bold', color='#1f2937', va='bottom')

    # Grade range + confidence
    grade_range = est.get('grade_range', '?')
    confidence  = est.get('confidence', '?')
    ax_r.text(0.0, 0.97, f"{grade_range}  •  {confidence} confidence",
              transform=ax_r.transAxes, fontsize=11, color='#374151', va='top')

    # Known grade marker
    if known_grade is not None:
        ax_r.text(1.0, 0.97, f'Known grade: PSA {known_grade}',
                  transform=ax_r.transAxes, fontsize=11, color='#4f46e5',
                  va='top', ha='right', fontstyle='italic')

    # Probability bars (PSA 10 → 1)
    bar_top = 0.88
    bar_h   = 0.055
    bar_gap = 0.012
    for g in range(10, 0, -1):
        prob  = dist.get(str(g), 0)
        color = GRADE_COLORS.get(g, '#6b7280')
        y     = bar_top - (10 - g) * (bar_h + bar_gap)

        # Background track
        ax_r.add_patch(mpatches.FancyBboxPatch(
            (0.12, y), 0.72, bar_h,
            boxstyle='round,pad=0.005',
            facecolor='#f3f4f6', edgecolor='none',
            transform=ax_r.transAxes, zorder=1))

        # Fill
        if prob > 0:
            ax_r.add_patch(mpatches.FancyBboxPatch(
                (0.12, y), 0.72 * prob, bar_h,
                boxstyle='round,pad=0.005',
                facecolor=color, alpha=0.85, edgecolor='none',
                transform=ax_r.transAxes, zorder=2))

        # Known grade marker line
        if known_grade == g:
            ax_r.axhline(y=y + bar_h/2, xmin=0.12, xmax=0.84,
                         color='#4f46e5', linewidth=2, linestyle='--',
                         transform=ax_r.transAxes, zorder=3)

        # Labels
        ax_r.text(0.10, y + bar_h/2, f'PSA {g}',
                  transform=ax_r.transAxes, fontsize=8.5,
                  va='center', ha='right', color='#374151')
        ax_r.text(0.86, y + bar_h/2, f'{prob*100:.0f}%',
                  transform=ax_r.transAxes, fontsize=8.5,
                  va='center', ha='left', color='#374151')

    # Issues
    issue_y = bar_top - 10 * (bar_h + bar_gap) - 0.04
    if issues:
        ax_r.text(0.0, issue_y, 'Issues detected:', transform=ax_r.transAxes,
                  fontsize=9, fontweight='bold', color='#92400e', va='top')
        for i, iss in enumerate(issues[:6]):
            ax_r.text(0.02, issue_y - 0.045*(i+1),
                      f'⚠ {textwrap.shorten(iss, 65)}',
                      transform=ax_r.transAxes, fontsize=8.5, color='#78350f', va='top')
    else:
        ax_r.text(0.0, issue_y, '✓ No significant issues detected',
                  transform=ax_r.transAxes, fontsize=9, color='#065f46', va='top')

    # Quality warnings
    warnings = quality.get('warnings', [])
    if warnings:
        warn_y = issue_y - 0.045 * (len(issues[:6]) + 1) - 0.02
        for w in warnings[:3]:
            ax_r.text(0.0, warn_y, f'ℹ {w}',
                      transform=ax_r.transAxes, fontsize=8, color='#6b7280', va='top')
            warn_y -= 0.04

    # Footer: cost + tokens
    ax_r.text(0.0, 0.01,
              f"Tokens: {usage.get('input_tokens',0):,} in / {usage.get('output_tokens',0):,} out  •  Cost: ${cost:.4f}",
              transform=ax_r.transAxes, fontsize=7.5, color='#9ca3af', va='bottom')

    plt.suptitle('Analyze Card — Local Evaluation', fontsize=14, y=1.01, color='#111827')
    plt.tight_layout()
    plt.show()
    return result

print('Visualisation functions defined ✓')

Visualisation functions defined ✓


## Test Cases

Edit `TEST_CASES` below — add the image paths and the **known PSA grade** (if you have it). Leave `known_grade=None` if you're just exploring.

In [5]:
# ── Define your test cases ───────────────────────────────────────
# Each entry: (list of image paths, listing title or description, known PSA grade or None)

TEST_CASES = [
    (
        ['cards/image0_front.jpeg', 'cards/image0_back.jpeg'],
        'Card 0',   # Replace with card name if known, e.g. "1999 Charizard Base Set Holo"
        None,       # Replace with known PSA grade, e.g. 9
    ),
    (
        ['cards/image1_front.jpeg', 'cards/image1_back.jpeg'],
        'Card 1',
        None,
    ),
    (
        ['cards/image2_front.jpeg', 'cards/image2_back.jpeg'],
        'Card 2',
        None,
    ),
]

print(f'{len(TEST_CASES)} test cases defined')

3 test cases defined


In [6]:
# ── Run all test cases ───────────────────────────────────────────
results = []
total_cost = 0.0

for i, (paths, title, known_grade) in enumerate(TEST_CASES[-1:]):
    print(f'\n{'='*60}')
    print(f'Card {i+1}/{len(TEST_CASES)}: {title}')
    if known_grade:
        print(f'Known grade: PSA {known_grade}')
    print('Analyzing...')

    try:
        result = analyze_card(paths, title)
        cost   = estimate_cost(result['_usage'])
        total_cost += cost
        results.append((result, paths, known_grade))

        display_result(result, paths, known_grade)

        print(f"Grade range : {result['grade_estimate']['grade_range']}")
        print(f"Confidence  : {result['grade_estimate']['confidence']}")
        print(f"Issues      : {len(result.get('issues', []))}")
        print(f"Cost        : ${cost:.4f}")

    except Exception as e:
        print(f'ERROR: {e}')

print(f'\n{"="*60}')
print(f'Total cost for {len(TEST_CASES)} cards: ${total_cost:.4f}')


Card 1/3: Card 2
Analyzing...
ERROR: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Could not process image'}, 'request_id': 'req_011Cb2mAPTEitFB6omuMoDDm'}

Total cost for 3 cards: $0.0000


In [7]:
# ── Accuracy summary (only for cards with known grades) ──────────
graded = [(r, kg) for r, _, kg in results if kg is not None]

if not graded:
    print('No known grades set — fill in known_grade in TEST_CASES to see accuracy metrics.')
else:
    print(f'{'Card':<20} {'Known':>6} {'Top Pred':>9} {'Peak%':>7} {'Correct?':>9}')
    print('-' * 57)

    correct_exact = 0
    correct_within1 = 0

    for result, known_grade in graded:
        dist    = result['grade_estimate']['distribution']
        top_g   = int(max(dist, key=lambda k: dist[k]))
        peak_p  = max(dist.values())
        name    = result['card_identity'].get('name', '?')[:19]
        exact   = top_g == known_grade
        within1 = abs(top_g - known_grade) <= 1

        if exact:   correct_exact   += 1
        if within1: correct_within1 += 1

        tick = '✓' if exact else ('~' if within1 else '✗')
        print(f'{name:<20} {known_grade:>6} {top_g:>9} {peak_p*100:>6.0f}% {tick:>9}')

    n = len(graded)
    print(f'\nExact match  : {correct_exact}/{n} ({correct_exact/n*100:.0f}%)')
    print(f'Within ±1    : {correct_within1}/{n} ({correct_within1/n*100:.0f}%)')

No known grades set — fill in known_grade in TEST_CASES to see accuracy metrics.


In [8]:
# ── Raw JSON (for debugging) ─────────────────────────────────────
# Uncomment to see the full response for any card
card_idx = 0   # 0-based index into results
print(json.dumps(results[card_idx][0], indent=2))

IndexError: list index out of range

In [ ]:
results